Importy bibliotek i instalacje

In [ ]:
import os
import importlib.util
import subprocess
import sys
from pathlib import Path

def install_if_missing(package_name, import_name=None):
    """
    Installs a package only if it is not already available.
    package_name: name used by pip
    import_name: name used in import
    """
    if import_name is None:
        import_name = package_name

    if importlib.util.find_spec(import_name) is None:
        print(f"{package_name} not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
    else:
        print(f"{package_name} is already installed.")


install_if_missing("ultralytics")
install_if_missing("kagglehub")

import kagglehub
import yaml
from ultralytics import YOLO

ultralytics is already installed.
kagglehub is already installed.


In [13]:
DATASET_NAME = "barkataliarbab/license-plate-detection-dataset-10125-images"

dataset_path = Path(kagglehub.dataset_download(DATASET_NAME))
print("Dataset path:", dataset_path)
print("Files:", list(dataset_path.iterdir())[:10])

Using Colab cache for faster access to the 'license-plate-detection-dataset-10125-images' dataset.
Dataset path: /kaggle/input/license-plate-detection-dataset-10125-images
Files: [PosixPath('/kaggle/input/license-plate-detection-dataset-10125-images/README.dataset.txt'), PosixPath('/kaggle/input/license-plate-detection-dataset-10125-images/README.roboflow.txt'), PosixPath('/kaggle/input/license-plate-detection-dataset-10125-images/data.yaml'), PosixPath('/kaggle/input/license-plate-detection-dataset-10125-images/valid'), PosixPath('/kaggle/input/license-plate-detection-dataset-10125-images/test'), PosixPath('/kaggle/input/license-plate-detection-dataset-10125-images/train')]


In [18]:
def find_dir(root, target_suffix):
    matches = [
        p for p in root.rglob("*")
        if p.is_dir() and str(p).replace("\\", "/").endswith(target_suffix)
    ]
    
    if not matches:
        raise FileNotFoundError(f"Could not find directory ending with: {target_suffix}")
    
    return matches[0]


train_images_dir = find_dir(dataset_path, "train/images")
valid_images_dir = find_dir(dataset_path, "valid/images")
test_images_dir = find_dir(dataset_path, "test/images")

train_labels_dir = Path(str(train_images_dir).replace("/images", "/labels"))
valid_labels_dir = Path(str(valid_images_dir).replace("/images", "/labels"))
test_labels_dir = Path(str(test_images_dir).replace("/images", "/labels"))

print("Train images:", train_images_dir)
print("Train labels:", train_labels_dir)
print("Valid images:", valid_images_dir)
print("Valid labels:", valid_labels_dir)
print("Test images:", test_images_dir)
print("Test labels:", test_labels_dir)

Train images: /kaggle/input/license-plate-detection-dataset-10125-images/train/images
Train labels: /kaggle/input/license-plate-detection-dataset-10125-images/train/labels
Valid images: /kaggle/input/license-plate-detection-dataset-10125-images/valid/images
Valid labels: /kaggle/input/license-plate-detection-dataset-10125-images/valid/labels
Test images: /kaggle/input/license-plate-detection-dataset-10125-images/test/images
Test labels: /kaggle/input/license-plate-detection-dataset-10125-images/test/labels


## MINI DATASET

In [19]:
import random
import shutil

def copy_yolo_subset(images_dir, labels_dir, output_images_dir, output_labels_dir, n_images, seed=42):
    random.seed(seed)
    
    output_images_dir.mkdir(parents=True, exist_ok=True)
    output_labels_dir.mkdir(parents=True, exist_ok=True)
    
    image_files = (
        list(images_dir.glob("*.jpg")) +
        list(images_dir.glob("*.jpeg")) +
        list(images_dir.glob("*.png"))
    )
    
    print(f"Found {len(image_files)} images in {images_dir}")
    
    random.shuffle(image_files)
    
    copied = 0
    
    for img_path in image_files:
        label_path = labels_dir / f"{img_path.stem}.txt"
        
        if label_path.exists():
            shutil.copy2(img_path, output_images_dir / img_path.name)
            shutil.copy2(label_path, output_labels_dir / label_path.name)
            copied += 1
        
        if copied >= n_images:
            break
    
    print(f"Copied {copied} images to {output_images_dir}")
    
    if copied == 0:
        raise RuntimeError("No images were copied. Check image/label folder paths.")


In [20]:
MINI_DATASET_DIR = Path("mini_license_plate_dataset")

# Remove old mini dataset if it exists, so you get a clean version
if MINI_DATASET_DIR.exists():
    shutil.rmtree(MINI_DATASET_DIR)

copy_yolo_subset(
    train_images_dir,
    train_labels_dir,
    MINI_DATASET_DIR / "train" / "images",
    MINI_DATASET_DIR / "train" / "labels",
    n_images=100
)

copy_yolo_subset(
    valid_images_dir,
    valid_labels_dir,
    MINI_DATASET_DIR / "valid" / "images",
    MINI_DATASET_DIR / "valid" / "labels",
    n_images=20
)

copy_yolo_subset(
    test_images_dir,
    test_labels_dir,
    MINI_DATASET_DIR / "test" / "images",
    MINI_DATASET_DIR / "test" / "labels",
    n_images=20
)

Found 7057 images in /kaggle/input/license-plate-detection-dataset-10125-images/train/images
Copied 100 images to mini_license_plate_dataset/train/images
Found 2048 images in /kaggle/input/license-plate-detection-dataset-10125-images/valid/images
Copied 20 images to mini_license_plate_dataset/valid/images
Found 1020 images in /kaggle/input/license-plate-detection-dataset-10125-images/test/images
Copied 20 images to mini_license_plate_dataset/test/images


In [21]:
mini_data_yaml = MINI_DATASET_DIR / "data.yaml"

mini_data_config = {
    "path": str(MINI_DATASET_DIR.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["License_Plate"]
}

with open(mini_data_yaml, "w") as f:
    yaml.safe_dump(mini_data_config, f, sort_keys=False)

print("Mini data.yaml created at:", mini_data_yaml)
print(yaml.safe_dump(mini_data_config, sort_keys=False))

Mini data.yaml created at: mini_license_plate_dataset/data.yaml
path: /content/mini_license_plate_dataset
train: train/images
val: valid/images
test: test/images
nc: 1
names:
- License_Plate



In [22]:
for split in ["train", "valid", "test"]:
    image_dir = MINI_DATASET_DIR / split / "images"
    label_dir = MINI_DATASET_DIR / split / "labels"
    
    print(split)
    print("images:", len(list(image_dir.glob("*"))))
    print("labels:", len(list(label_dir.glob("*"))))

train
images: 100
labels: 100
valid
images: 20
labels: 20
test
images: 20
labels: 20


In [23]:
model = YOLO("yolo11n.pt")

results = model.train(
    data=str(mini_data_yaml),
    epochs=3,
    imgsz=320,
    batch=4,
    project="license_plate_detection",
    name="mini_test",
    seed=42
)

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=mini_license_plate_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=mini_test-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

In [3]:
# FULL RUN

model = YOLO("yolo11n.pt")

results = model.train(
    data=data_yaml,
    epochs=30,
    imgsz=640,
    batch=16,
    project="license_plate_detection",
    name="yolo11n_baseline",
    seed=42,
    patience=10,
    plots=True
)


Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/license-plate-detection-dataset-10125-images/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_baseline-2, nbs=64, nms=False, opset=None, optimize=False, optimiz

KeyboardInterrupt: 

walidacja

In [ ]:
best_model = YOLO("license_plate_detection/yolo11n_baseline/weights/best.pt")

metrics = best_model.val()
metrics

Predykcja dla przykładowego zdjęcia

In [ ]:
results = best_model.predict(
    source="path_to_some_test_image.jpg",
    conf=0.25,
    save=True
)